# 01 · Bronze ingestion
**Phase 3 – Bronze layer.** Loads the 10 raw Parquet folders from the `source` container into Delta tables in `glovo_dev.bronze` using **Auto Loader**.

**Rules for Bronze**
- Data is stored **exactly as received**: no cleaning, no type changes.
- Each row gets audit columns: when it was loaded and from which file.
- Re-running the notebook only loads **new** files (incremental).

## 1. Checkpoint storage
A Unity Catalog volume to hold Auto Loader's checkpoints and schema files (which files were already loaded).

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS glovo_dev.bronze.checkpoints
COMMENT 'Auto Loader checkpoints and schema locations for bronze ingestion';

## 2. Settings
paths and names used by every ingestion below, defined once in one place.

In [0]:
from pyspark.sql import functions as F

SOURCE = "abfss://source@stglovolake122.dfs.core.windows.net"
CATALOG = "glovo_dev"
SCHEMA = "bronze"
CHECKPOINTS = "/Volumes/glovo_dev/bronze/checkpoints"

tables = [
    "cities", "couriers", "cuisine_category_translation", "customers", "menu_items",
    "order_items", "orders", "payments", "restaurants", "reviews",
]

## 3. Load all 10 tables
### 3a. Define the Auto Loader reads
Prepares one streaming read per table (lazy: no data is loaded yet).

In [0]:
def read_source(table):
    return (
        spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", "parquet")
            .option("cloudFiles.schemaLocation", f"{CHECKPOINTS}/{table}/_schema")
            .load(f"{SOURCE}/{table}/")
            .withColumn("_ingested_at", F.current_timestamp())
            .withColumn("_source_file", F.col("_metadata.file_path"))
    )

streams = {t: read_source(t) for t in tables}

### 3b. Write to Bronze tables
Runs each read and saves it to `glovo_dev.bronze.<table>`. Already-loaded files are skipped thanks to the checkpoint.

In [0]:
def write_bronze(table, df):
    (
        df.writeStream
            .option("checkpointLocation", f"{CHECKPOINTS}/{table}/_checkpoint")
            .trigger(availableNow=True)
            .toTable(f"{CATALOG}.{SCHEMA}.{table}")
            .awaitTermination()
    )

for t, df in streams.items():
    write_bronze(t, df)
    print(f"loaded {t}")

In [0]:
for t in tables:
    n = spark.table(f"{CATALOG}.{SCHEMA}.{t}").count()
    print(f"{t:30} {n:>9,} rows")